In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 

In [6]:
%%writefile centralized_distilbert_imdb.py
import os
import re
import json
import time
import random
import argparse

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: str, default=None):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params


def get_latest_checkpoint(checkpoint_root: str):
    if not os.path.exists(checkpoint_root):
        return None
    candidates = []
    for name in os.listdir(checkpoint_root):
        full = os.path.join(checkpoint_root, name)
        if os.path.isdir(full):
            m = re.match(r"^checkpoint-epoch-(\d+)$", name)
            if m:
                candidates.append((int(m.group(1)), full))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def cleanup_old_checkpoints(checkpoint_root: str, keep_last_n: int = 2):
    if not os.path.exists(checkpoint_root):
        return
    candidates = []
    for name in os.listdir(checkpoint_root):
        full = os.path.join(checkpoint_root, name)
        if os.path.isdir(full):
            m = re.match(r"^checkpoint-epoch-(\d+)$", name)
            if m:
                candidates.append((int(m.group(1)), full))
    candidates.sort(key=lambda x: x[0])
    to_delete = candidates[:-keep_last_n]
    for _, ckpt_path in to_delete:
        for root, dirs, files in os.walk(ckpt_path, topdown=False):
            for file in files:
                os.remove(os.path.join(root, file))
            for d in dirs:
                os.rmdir(os.path.join(root, d))
        os.rmdir(ckpt_path)


def save_checkpoint(
    checkpoint_root,
    epoch,
    model,
    tokenizer,
    optimizer,
    scheduler,
    history,
    state_dict_to_save,
):
    ckpt_dir = os.path.join(checkpoint_root, f"checkpoint-epoch-{epoch}")
    ensure_dir(ckpt_dir)

    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
    torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
    save_json(history, os.path.join(ckpt_dir, "history.json"))
    save_json(state_dict_to_save, os.path.join(ckpt_dir, "trainer_state.json"))

    cleanup_old_checkpoints(checkpoint_root, keep_last_n=2)
    return ckpt_dir


def evaluate_model(model, dataloader, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            losses.append(loss.item())
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch["labels"].cpu().numpy().tolist())

    eval_loss = float(np.mean(losses)) if losses else 0.0
    accuracy = float(accuracy_score(all_labels, all_preds))
    macro_f1 = float(f1_score(all_labels, all_preds, average="macro"))

    return {
        "eval_loss": eval_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    }


def train_one_epoch(model, dataloader, optimizer, scheduler, device, use_amp=False):
    model.train()
    losses = []

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if torch.cuda.is_available() else None

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler is not None:
            with torch.amp.autocast("cuda", enabled=True):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

        scheduler.step()
        losses.append(loss.item())

    train_loss = float(np.mean(losses)) if losses else 0.0
    return train_loss


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased")
    parser.add_argument("--dataset_name", type=str, default="imdb")
    parser.add_argument("--output_dir", type=str, required=True)

    parser.add_argument("--max_length", type=int, default=256)
    parser.add_argument("--train_batch_size", type=int, default=16)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--warmup_ratio", type=float, default=0.1)

    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--metric_for_best_model", type=str, default="accuracy", choices=["accuracy", "macro_f1"])
    parser.add_argument("--val_size", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)

    args = parser.parse_args()

    set_seed(args.seed)

    output_dir = args.output_dir
    checkpoint_root = os.path.join(output_dir, "checkpoints")
    best_model_dir = os.path.join(output_dir, "best_model")
    final_model_dir = os.path.join(output_dir, "final_model")
    logs_dir = os.path.join(output_dir, "logs")

    ensure_dir(output_dir)
    ensure_dir(checkpoint_root)
    ensure_dir(best_model_dir)
    ensure_dir(final_model_dir)
    ensure_dir(logs_dir)

    history_json_path = os.path.join(logs_dir, "history.json")
    history_csv_path = os.path.join(logs_dir, "history.csv")
    run_info_json_path = os.path.join(output_dir, "run_info.json")

    print("=" * 100)
    print("Centralized DistilBERT (full fine-tuning) on IMDB")
    print(f"Output directory: {output_dir}")
    print(f"Checkpoints dir : {checkpoint_root}")
    print(f"Best model dir  : {best_model_dir}")
    print(f"Final model dir : {final_model_dir}")
    print(f"Logs dir        : {logs_dir}")
    print("=" * 100)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()
    print(f"Device: {device} | AMP: {use_amp}")

    latest_ckpt = get_latest_checkpoint(checkpoint_root)

    if latest_ckpt is not None:
        print(f"Resuming from latest checkpoint: {latest_ckpt}")
        tokenizer = AutoTokenizer.from_pretrained(latest_ckpt, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(latest_ckpt, num_labels=2)
        trainer_state = load_json(os.path.join(latest_ckpt, "trainer_state.json"), default={})
        history = load_json(os.path.join(latest_ckpt, "history.json"), default=[])
        start_epoch = int(trainer_state.get("epoch", 0)) + 1
        best_metric_so_far = float(trainer_state.get("best_metric_so_far", -1.0))
        patience_counter = int(trainer_state.get("patience_counter", 0))
    else:
        print("No checkpoint found. Starting from scratch.")
        tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=2)
        history = []
        start_epoch = 1
        best_metric_so_far = -1.0
        patience_counter = 0

    model.to(device)

    trainable_params, total_params = count_parameters(model)
    setting = "centralized_full_finetuning"

    raw_dataset = load_dataset(args.dataset_name)

    train_valid = raw_dataset["train"].train_test_split(
        test_size=args.val_size,
        seed=args.seed,
        stratify_by_column="label"
    )
    train_ds = train_valid["train"]
    val_ds = train_valid["test"]
    test_ds = raw_dataset["test"]

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=args.max_length,
            padding=False
        )

    train_ds = train_ds.map(tokenize_fn, batched=True, desc="Tokenizing train")
    val_ds = val_ds.map(tokenize_fn, batched=True, desc="Tokenizing val")
    test_ds = test_ds.map(tokenize_fn, batched=True, desc="Tokenizing test")

    keep_cols = ["input_ids", "attention_mask", "label"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols]).rename_column("label", "labels")
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols]).rename_column("label", "labels")
    test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep_cols]).rename_column("label", "labels")

    train_ds.set_format(type="torch")
    val_ds.set_format(type="torch")
    test_ds.set_format(type="torch")

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    train_loader = DataLoader(
        train_ds,
        batch_size=args.train_batch_size,
        shuffle=True,
        collate_fn=data_collator,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.eval_batch_size,
        shuffle=False,
        collate_fn=data_collator,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=args.eval_batch_size,
        shuffle=False,
        collate_fn=data_collator,
        pin_memory=torch.cuda.is_available(),
    )

    steps_per_epoch = len(train_loader)
    if steps_per_epoch == 0:
        raise ValueError("Train DataLoader is empty. Please check dataset preprocessing.")

    virtual_max_epochs = 1000
    total_training_steps = steps_per_epoch * virtual_max_epochs
    warmup_steps = int(total_training_steps * args.warmup_ratio)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps,
    )

    if latest_ckpt is not None:
        opt_path = os.path.join(latest_ckpt, "optimizer.pt")
        sch_path = os.path.join(latest_ckpt, "scheduler.pt")
        if os.path.exists(opt_path):
            optimizer.load_state_dict(torch.load(opt_path, map_location=device))
        if os.path.exists(sch_path):
            scheduler.load_state_dict(torch.load(sch_path, map_location=device))

    run_info = {
        "task_type": "text_classification",
        "learning_type": "centralized",
        "model_name": args.model_name,
        "dataset_name": args.dataset_name,
        "setting": setting,
        "output_dir": output_dir,
        "checkpoints_dir": checkpoint_root,
        "best_model_dir": best_model_dir,
        "final_model_dir": final_model_dir,
        "logs_dir": logs_dir,
        "history_json": history_json_path,
        "history_csv": history_csv_path,
        "client_history_json": None,
        "client_history_csv": None,
        "note": "Centralized training, so client logs are not generated."
    }
    save_json(run_info, run_info_json_path)

    print(f"Train samples: {len(train_ds)}")
    print(f"Val samples  : {len(val_ds)}")
    print(f"Test samples : {len(test_ds)}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params    : {total_params:,}")
    print(f"Metric for best : {args.metric_for_best_model}")
    print(f"Early stopping patience: {args.patience}")
    print(f"Starting from epoch: {start_epoch}")

    stopped_early = False
    last_completed_epoch = start_epoch - 1
    epoch = start_epoch

    while True:
        print("\n" + "=" * 100)
        print(f"Epoch {epoch}")
        print("=" * 100)

        epoch_start_time = time.time()

        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            use_amp=use_amp,
        )

        eval_metrics = evaluate_model(model, val_loader, device=device)

        epoch_time = time.time() - epoch_start_time
        current_metric = float(eval_metrics[args.metric_for_best_model])
        is_new_best = current_metric > best_metric_so_far

        if is_new_best:
            best_metric_so_far = current_metric
            patience_counter = 0
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)
        else:
            patience_counter += 1

        row = {
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "eval_loss": float(eval_metrics["eval_loss"]),
            "accuracy": float(eval_metrics["accuracy"]),
            "macro_f1": float(eval_metrics["macro_f1"]),
            "time_per_epoch": float(epoch_time),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": str(args.model_name),
            "dataset_name": str(args.dataset_name),
            "setting": str(setting),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }
        history.append(row)

        pd.DataFrame(history).to_csv(history_csv_path, index=False)
        save_json(history, history_json_path)

        trainer_state = {
            "epoch": int(epoch),
            "best_metric_so_far": float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "metric_for_best_model": args.metric_for_best_model,
            "stopped_early": False,
        }

        ckpt_dir = save_checkpoint(
            checkpoint_root=checkpoint_root,
            epoch=epoch,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            state_dict_to_save=trainer_state,
        )

        print(f"Epoch {epoch} completed.")
        print(f"train_loss         = {row['train_loss']:.6f}")
        print(f"eval_loss          = {row['eval_loss']:.6f}")
        print(f"accuracy           = {row['accuracy']:.6f}")
        print(f"macro_f1           = {row['macro_f1']:.6f}")
        print(f"time_per_epoch     = {row['time_per_epoch']:.2f} sec")
        print(f"best_metric_so_far = {row['best_metric_so_far']:.6f}")
        print(f"patience_counter   = {row['patience_counter']}")
        print(f"is_new_best        = {row['is_new_best']}")
        print(f"Saved checkpoint   = {ckpt_dir}")

        last_completed_epoch = epoch

        if patience_counter >= args.patience:
            print(f"Early stopping triggered: no improvement for {args.patience} consecutive epoch(s).")
            stopped_early = True
            break

        epoch += 1

    model.save_pretrained(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
    best_model.to(device)

    best_val_metrics = evaluate_model(best_model, val_loader, device=device)
    best_test_metrics = evaluate_model(best_model, test_loader, device=device)

    final_summary = {
        "model_name": args.model_name,
        "dataset_name": args.dataset_name,
        "setting": setting,
        "output_dir": output_dir,
        "checkpoints_dir": checkpoint_root,
        "best_model_dir": best_model_dir,
        "final_model_dir": final_model_dir,
        "logs_dir": logs_dir,
        "history_json": history_json_path,
        "history_csv": history_csv_path,
        "client_history_json": None,
        "client_history_csv": None,
        "total_epochs_completed": int(last_completed_epoch),
        "early_stopped": bool(stopped_early),
        "metric_for_best_model": args.metric_for_best_model,
        "best_metric_so_far": float(best_metric_so_far),
        "best_validation_metrics": {
            "eval_loss": float(best_val_metrics["eval_loss"]),
            "accuracy": float(best_val_metrics["accuracy"]),
            "macro_f1": float(best_val_metrics["macro_f1"]),
        },
        "best_test_metrics": {
            "eval_loss": float(best_test_metrics["eval_loss"]),
            "accuracy": float(best_test_metrics["accuracy"]),
            "macro_f1": float(best_test_metrics["macro_f1"]),
        },
        "history_columns": [
            "epoch",
            "train_loss",
            "eval_loss",
            "accuracy",
            "macro_f1",
            "time_per_epoch",
            "trainable_params",
            "total_params",
            "model_name",
            "dataset_name",
            "setting",
            "best_metric_so_far",
            "patience_counter",
            "is_new_best",
        ],
        "note": "This is centralized learning. Therefore no client_history.json/csv is generated.",
    }
    save_json(final_summary, os.path.join(output_dir, "final_summary.json"))

    print("\n" + "=" * 100)
    print("TRAINING FINISHED")
    print("=" * 100)
    print(f"Output dir         : {output_dir}")
    print(f"Checkpoints dir    : {checkpoint_root}")
    print(f"Best model dir     : {best_model_dir}")
    print(f"Final model dir    : {final_model_dir}")
    print(f"Logs dir           : {logs_dir}")
    print(f"history.json       : {history_json_path}")
    print(f"history.csv        : {history_csv_path}")
    print(f"run_info.json      : {run_info_json_path}")
    print(f"final_summary.json : {os.path.join(output_dir, 'final_summary.json')}")
    print("Generated files/folders:")
    print("- checkpoints/checkpoint-epoch-* (only 2 latest checkpoints kept)")
    print("- best_model/")
    print("- final_model/")
    print("- logs/history.json")
    print("- logs/history.csv")
    print("- run_info.json")
    print("- final_summary.json")
    print("\nSelf-check:")
    print("✓ Full runnable Python script")
    print("✓ Save all outputs to Google Drive/MyDrive")
    print("✓ Automatic resume from latest checkpoint")
    print("✓ Keep only 2 latest checkpoints")
    print("✓ Early stopping with patience=3")
    print("✓ Save best_model and final_model separately")
    print("✓ Export history.json and history.csv")
    print("✓ History columns are complete")
    print("✓ No client logs because this is centralized, not federated")


if __name__ == "__main__":
    main()

Overwriting centralized_distilbert_imdb.py


In [7]:
!python centralized_distilbert_imdb.py \
  --model_name distilbert-base-uncased \
  --dataset_name imdb \
  --output_dir "/content/drive/MyDrive/centralized_distilbert_imdb"

Centralized DistilBERT (full fine-tuning) on IMDB
Output directory: /content/drive/MyDrive/centralized_distilbert_imdb
Checkpoints dir : /content/drive/MyDrive/centralized_distilbert_imdb/checkpoints
Best model dir  : /content/drive/MyDrive/centralized_distilbert_imdb/best_model
Final model dir : /content/drive/MyDrive/centralized_distilbert_imdb/final_model
Logs dir        : /content/drive/MyDrive/centralized_distilbert_imdb/logs
Device: cuda | AMP: True
No checkpoint found. Starting from scratch.
Loading weights: 100% 100/100 [00:00<00:00, 20817.47it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
